# Sheet 2.1: Attention and self-attention

**Roadmap topic 2.1** (Fanout 06.01, 06.02). Read/watch first: *The Illustrated Transformer*, 3Blue1Brown's attention chapter, *Attention Is All You Need* (all of it), *The Annotated Transformer*.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Do not call `F.scaled_dot_product_attention` or `nn.MultiheadAttention` in your code (the checks use them as references). CPU is fine for the checks; Exercise 5 needs a Colab T4.

In [ ]:
import math, time, torch, torch.nn.functional as F
torch.manual_seed(0)

## Exercise 1: Multi-head attention with causal and padding masks

`mha(x, Wq, Wk, Wv, Wo, n_heads, causal, pad_mask)`:
- `x (B, T, C)`, weights `(C, C)` used as `x @ W`.
- `pad_mask (B, T)` bool, `True` = real token, `False` = padding (right-padded). Padded **keys** must get zero attention.
- `causal=True`: position `t` attends only to `≤ t`.
- Return `(B, T, C)`. Use the max-subtraction trick (or `masked_fill` with `-inf` then softmax) and make sure no row is all `-inf` for real query positions.

In [ ]:
def mha(x, Wq, Wk, Wv, Wo, n_heads, causal=True, pad_mask=None):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    torch.set_default_dtype(torch.float64)
    B, T, C, H = 3, 9, 32, 4
    x = torch.randn(B, T, C); W = [torch.randn(C, C) / math.sqrt(C) for _ in range(4)]
    lengths = torch.tensor([9, 5, 1]); pad = torch.arange(T)[None] < lengths[:, None]
    def ref(causal, pad_mask):
        q, k, v = [(x @ w).view(B, T, H, C // H).transpose(1, 2) for w in W[:3]]
        m = torch.ones(T, T, dtype=torch.bool)
        if causal: m = torch.tril(m)
        m = m[None, None].expand(B, 1, T, T)
        if pad_mask is not None: m = m & pad_mask[:, None, None, :]
        return F.scaled_dot_product_attention(q, k, v, attn_mask=m).transpose(1, 2).reshape(B, T, C) @ W[3]
    for causal in (True, False):
        for pm in (None, pad):
            out = mha(x, *W, H, causal=causal, pad_mask=pm); r = ref(causal, pm)
            valid = pad if pm is not None else torch.ones(B, T, dtype=torch.bool)
            assert torch.isfinite(out[valid]).all(), "NaN/inf at real positions"
            assert torch.allclose(out[valid], r[valid], atol=1e-10), f"mismatch causal={causal} pad={pm is not None}"
    out2 = mha(torch.cat([x[:, :5], torch.randn(B, 4, C)], 1), *W, H, causal=True)
    assert torch.allclose(out2[:, :5], mha(x, *W, H, causal=True)[:, :5]), "causal leak: future tokens change the past"
    torch.set_default_dtype(torch.float32)
    print("PASS")
_check1()

## Exercise 2: Why divide by √d?

`attn_entropy(d, scaled, n=64, trials=200)`: for random `q, k ∈ R^d` with i.i.d. N(0,1) entries, compute softmax over `n` keys of `q·k` (divided by `√d` if `scaled`), and return the mean entropy in nats. Also `max_grad_ratio(d, scaled)`: the mean over trials of `‖∂softmax/∂scores‖_F` (use your Jacobian from sheet 0.4), which shows the vanishing-gradient side of saturation.

In [ ]:
def attn_entropy(d, scaled, n=64, trials=200):
    # YOUR CODE
    raise NotImplementedError
def softmax_jac_norm(d, scaled, n=64, trials=200):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    torch.manual_seed(0)
    ds = [16, 64, 256, 1024]
    hs = [attn_entropy(d, True) for d in ds]; hu = [attn_entropy(d, False) for d in ds]
    assert max(hs) - min(hs) < 0.15, f"scaled entropy should not depend on d: {hs}"
    assert all(a > b for a, b in zip(hu, hu[1:])) and hu[-1] < 0.3, f"unscaled entropy should collapse: {hu}"
    js, ju = softmax_jac_norm(1024, True), softmax_jac_norm(1024, False)
    assert ju < js / 2, "unscaled softmax should have much smaller gradients when saturated"
    print("PASS  entropy scaled:", [round(h, 2) for h in hs], " unscaled:", [round(h, 2) for h in hu])
_check2()

**Written (2a):** Show that `Var(q·k) = d` for i.i.d. unit-variance entries, and connect your numbers to it. Why do some modern models add QK-norm (normalising q and k) on top of the scale?

**Your answer:**

## Exercise 3: KV cache for incremental decoding

`CachedSelfAttention` (single layer, causal, multi-head, no padding) must give **the same outputs** whether you run the full sequence once (`forward_full`) or feed tokens one at a time (`step`), appending keys/values to a cache. Then report the speedup for generating 512 tokens.

In [ ]:
class CachedSelfAttention(torch.nn.Module):
    def __init__(self, C, n_heads):
        super().__init__()
        self.h = n_heads
        self.qkv = torch.nn.Linear(C, 3 * C, bias=False); self.o = torch.nn.Linear(C, C, bias=False)
        self.reset_cache()
    def reset_cache(self):
        # YOUR CODE
        raise NotImplementedError
    def forward_full(self, x):            # x (B, T, C) -> (B, T, C)
        # YOUR CODE
        raise NotImplementedError
    def step(self, x_t):                  # x_t (B, 1, C) -> (B, 1, C), uses and updates the cache
        # YOUR CODE
        raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
@torch.no_grad()
def _check3():
    torch.manual_seed(0); m = CachedSelfAttention(64, 4).double(); x = torch.randn(2, 30, 64, dtype=torch.float64)
    full = m.forward_full(x); m.reset_cache()
    inc = torch.cat([m.step(x[:, t:t + 1]) for t in range(30)], 1)
    assert torch.allclose(full, inc, atol=1e-12), "cached outputs differ from full recomputation"
    m = CachedSelfAttention(256, 8); x = torch.randn(1, 512, 256)
    t0 = time.perf_counter()
    for T in range(1, 257): m.forward_full(x[:, :T])
    t_full = time.perf_counter() - t0
    m.reset_cache(); t0 = time.perf_counter()
    for T in range(256): m.step(x[:, T:T + 1])
    t_cache = time.perf_counter() - t0
    assert t_full / t_cache > 2, f"cache only {t_full / t_cache:.1f}x faster"
    print(f"PASS  256-token generation: {t_full / t_cache:.1f}x faster with the KV cache")
_check3()

## Exercise 4: Count the cost

For one attention layer with model width `C`, `H` heads, sequence length `T`, batch `B` (count a multiply-add as 2 FLOPs, forward pass only):
- `attn_flops(B, T, C)`: FLOPs of the QKV projection, the `QKᵀ` scores, `P·V`, and the output projection, summed.
- `score_matrix_bytes(B, H, T, bytes_per_el)`: memory of the materialised attention probabilities.
- `crossover_T(C)`: the sequence length at which the two `T²` terms equal the projection FLOPs.

In [ ]:
def attn_flops(B, T, C): raise NotImplementedError
def score_matrix_bytes(B, H, T, bytes_per_el): raise NotImplementedError
def crossover_T(C): raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    assert attn_flops(1, 1024, 768) == 2 * 1024 * 768 * 4 * 768 + 4 * 1024 ** 2 * 768, "attn_flops"
    assert score_matrix_bytes(8, 12, 4096, 2) == 8 * 12 * 4096 ** 2 * 2
    assert crossover_T(768) == 1536
    print(f"PASS  GPT-2 small at T=8192: probs take {score_matrix_bytes(1, 12, 8192, 2) / 2**30:.1f} GiB per layer per sequence in fp16")
_check4()

### Exercise 5 (Colab T4): find the wall
On a T4 in fp16, with `B=1, H=12, C=768`, double `T` from 512 until your `mha` runs out of memory (catch `torch.cuda.OutOfMemoryError`). Repeat with `F.scaled_dot_product_attention` (memory-efficient backend). Report the max `T` and the time per call for both, and explain the difference with Exercise 4's numbers.

In [ ]:
# Colab: OOM search

## Checkpoint
From a blank file, write multi-head causal attention with padding support in under 20 minutes, then pass Check 1 with it. Read Anthropic's *A Mathematical Framework for Transformer Circuits* afterwards and explain what the "QK circuit" and "OV circuit" are in your code.

---
### Hints for self-grading
- 2a: `q·k = Σ q_i k_i`, a sum of `d` independent products each with variance 1, so the variance is `d` and the std `√d`. Unscaled scores grow with `d`, the softmax saturates to one-hot, and its Jacobian vanishes. QK-norm bounds the logits even when q/k norms drift during training, preventing attention-logit growth instabilities.